<img src="https://raw.githubusercontent.com/OmarLi-360/excel-corporativo-12-claves/main/assets/logo-drbits.jpg" alt="Dr. Bits" width="170">

# Clave 8 · Limpieza reproducible
**Excel Corporativo · 12 Claves** — Power Query: los pasos quedan guardados · ⏱ 30 min

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmarLi-360/excel-corporativo-12-claves/blob/main/notebooks/08_limpieza_reproducible.ipynb) &nbsp; [Volver al tablero](https://omarli-360.github.io/excel-corporativo-12-claves/#clave-08)

**Al terminar podrás:**
1. Ver cómo cada paso de limpieza reduce los problemas de la exportación.
2. Medir cuántas filas cambia cada paso.
3. Estimar el tiempo que se ahorra cuando la limpieza se repite sola cada mes.

---
### Cómo usar este cuaderno (no necesitas programar)
1. **Guarda tu copia:** *Archivo › Guardar una copia en Drive*. Así tus cambios no se pierden.
2. **Ejecuta todo:** *Entorno de ejecución › Ejecutar todas* (Ctrl+F9). Si Colab avisa que el cuaderno no es de Google, elige *Ejecutar de todos modos*.
3. **Juega con los controles:** las celdas con listas desplegables a la derecha se pueden cambiar. Elige una opción y pulsa el botón ▶ de esa celda para redibujar.

El código está oculto a propósito: aquí importa entender el dato y el gráfico. Si tienes curiosidad, haz doble clic en una celda para verlo.

In [ ]:
# @title Preparar el entorno (se ejecuta sola con «Ejecutar todas»)
import os, sys, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown, HTML
warnings.filterwarnings("ignore")

EN_COLAB = "google.colab" in sys.modules
REPO = "OmarLi-360/excel-corporativo-12-claves"
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/main/data/"
BASE_LOCAL = "../data/"

def ruta(nombre):
    local = os.path.join(BASE_LOCAL, nombre)
    return local if os.path.exists(local) else BASE_URL + nombre

# --- formatos colombianos -------------------------------------------------
def miles(x):    return f"{x:,.0f}".replace(",", ".")
def pesos(x):    return "$ " + miles(x)
def millones(x): return "$ " + f"{x/1e6:,.0f}".replace(",", ".") + " M"
def porc(v, signo=False):
    t = f"{v:+.1%}" if signo else f"{v:.1%}"
    return t.replace(".", ",").replace("%", " %")
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
MM = "Millones de pesos"

# --- plantilla de gráficos Dr. Bits ----------------------------------------
AZUL, TEAL, MAGENTA, GRIS = "#4A5AD8", "#0E9BB8", "#D0336F", "#9AA3BF"
BIEN, MAL = "#15784A", "#B42318"
ejes = dict(gridcolor="#E4E8F2", zeroline=False, linecolor="#C9D0E2", exponentformat="none", separatethousands=True)
pio.templates["drbits"] = go.layout.Template(layout=dict(
    font=dict(family="Source Sans 3, Segoe UI, Arial, sans-serif", size=14, color="#11153A"),
    title=dict(font=dict(size=18, color="#00006A"), x=0, xanchor="left"),
    colorway=[AZUL, TEAL, MAGENTA], plot_bgcolor="white", paper_bgcolor="white",
    xaxis=ejes, yaxis=ejes, separators=",.", hoverlabel=dict(font_size=13),
    margin=dict(t=80, l=10, r=30, b=40)))
pio.templates.default = "plotly_white+drbits"

def eje_meses(fig, fechas, cada=None):
    """Etiquetas de meses en español en ejes de tiempo (Plotly las pone en inglés)."""
    f = pd.to_datetime(pd.Series(sorted(set(pd.to_datetime(list(fechas))))))
    paso = cada or (1 if len(f) <= 13 else 3)
    f = f.iloc[::paso]
    fig.update_xaxes(tickvals=list(f), ticktext=[f"{MESES[d.month - 1]} {d.year}" for d in f])

def dar_aire(fig, maximo, eje="x"):
    (fig.update_xaxes if eje == "x" else fig.update_yaxes)(range=[0, maximo * 1.22])

# --- revisor de retos -----------------------------------------------------
def revisar(respuesta, esperado, explicacion=""):
    if respuesta == "Elige una opción":
        display(Markdown("**Elige una opción en la lista de la derecha** y vuelve a ejecutar esta celda con ▶."))
    elif str(respuesta) == str(esperado):
        display(Markdown(f"✅ **Correcto.** {explicacion}"))
    else:
        display(Markdown(f"❌ **Todavía no.** Revisa el gráfico de arriba e inténtalo otra vez. *(Elegiste: {respuesta})*"))

# --- datos de Distribuidora Andina ----------------------------------------
ventas = pd.read_csv(ruta("ventas.csv"), parse_dates=["fecha"])
clientes = pd.read_csv(ruta("clientes.csv"))
productos = pd.read_csv(ruta("productos.csv"))
vendedores = pd.read_csv(ruta("vendedores.csv"))
sedes = pd.read_csv(ruta("sedes.csv"))
metas = pd.read_csv(ruta("metas.csv"))
df = (ventas
      .merge(sedes[["id_sede", "ciudad"]], on="id_sede", validate="many_to_one")
      .merge(productos[["sku", "producto", "categoria"]], on="sku", validate="many_to_one")
      .merge(clientes[["id_cliente", "razon_social", "segmento"]], on="id_cliente", validate="many_to_one")
      .merge(vendedores[["id_vendedor", "nombre_vendedor"]], on="id_vendedor", validate="many_to_one")
      .assign(anio=lambda d: d.fecha.dt.year, mes=lambda d: d.fecha.dt.month))
validas = df[df["estado"] != "Anulada"]          # las anuladas no son venta
ORDEN_CIUDADES = (validas[validas.anio == 2025].groupby("ciudad")["subtotal"].sum()
                  .sort_values(ascending=False).index.tolist())
display(Markdown(f"Datos listos: **{miles(len(ventas))} líneas de venta**, {len(clientes)} clientes, {len(productos)} productos. "
                 f"Fuente: {'archivos locales' if os.path.exists(BASE_LOCAL) else 'GitHub'}."))

## 1 · Los pasos aplicados
En Power Query cada acción queda registrada como un paso. Este cuaderno aplica los mismos pasos que haces en Excel sobre la hoja **ERP_Sucio**, uno detrás de otro, para que veas qué arregla cada uno:

1. Recortar y MAYÚSCULAS en `Cod Cliente`.
2. Reemplazar «$» y «.» en `Valor Neto` y convertir a número.
3. Quitar « und» en `Cantidad` y convertir a número.
4. Convertir `Fecha` con `try Date.FromText([Fecha], "es-CO") otherwise Date.From(Number.From([Fecha]))`.
5. Quitar duplicados.

Elige cuántos pasos aplicar, como si activaras las casillas del panel **Pasos aplicados**.

In [ ]:
# @title Definir los pasos de limpieza (los mismos de Power Query)
def problemas(t):
    cod = t["Cod Cliente"].astype(str)
    return pd.Series({
        "Código con espacios o minúsculas": int((cod != cod.str.strip().str.upper()).sum()),
        "Monto escrito como texto": int(t["Valor Neto"].map(lambda x: isinstance(x, str)).sum()),
        "Cantidad escrita como texto": int(t["Cantidad"].map(lambda x: isinstance(x, str)).sum()),
        "Fecha que no es fecha": int(t["Fecha"].map(lambda x: not isinstance(x, pd.Timestamp)).sum()),
        "Fila duplicada": int(t.duplicated().sum()),
    })

def a_fecha(s):
    s = str(s).strip()
    try:
        return pd.to_datetime(s, format="%d/%m/%Y")
    except ValueError:
        pass
    try:
        return pd.to_datetime(s, format="%Y-%m-%d")
    except ValueError:
        return pd.Timestamp("1899-12-30") + pd.Timedelta(days=int(float(s)))

def a_numero(x):
    if isinstance(x, str):
        x = x.replace("$", "").replace(".", "").replace("und", "").strip()
    return pd.to_numeric(x, errors="coerce")

PASOS = [
    ("Recortar y MAYÚSCULAS en Cod Cliente", lambda t: t.assign(**{"Cod Cliente": t["Cod Cliente"].astype(str).str.strip().str.upper()})),
    ("Limpiar Valor Neto ($ y puntos)",   lambda t: t.assign(**{"Valor Neto": t["Valor Neto"].map(a_numero)})),
    ("Limpiar Cantidad (quitar und)",            lambda t: t.assign(Cantidad=t["Cantidad"].map(a_numero))),
    ("Convertir Fecha (try … otherwise)",    lambda t: t.assign(Fecha=t["Fecha"].map(a_fecha))),
    ("Quitar duplicados",                    lambda t: t.drop_duplicates()),
]

def aplicar_pasos(origen, n=len(PASOS)):
    """Devuelve la tabla después de los primeros n pasos y la evolución de problemas y filas."""
    t = origen.copy()
    evolucion = [("Origen", problemas(t), len(t))]
    for nombre, paso in PASOS[:n]:
        t = paso(t)
        evolucion.append((nombre, problemas(t), len(t)))
    return t, evolucion
erp = pd.read_excel(ruta("Distribuidora_Andina_practica.xlsx"), sheet_name="ERP_Sucio", dtype=object)
display(Markdown(f"Hoja ERP_Sucio leída: **{miles(len(erp))} filas** y {erp.shape[1]} columnas."))

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Cuántos problemas quedan después de cada paso? |
> | **Qué se grafica** | Barras apiladas: problemas por tipo, desde el origen hasta el último paso aplicado |
> | **Por qué esta forma** | Muestra el avance total y qué tipo de problema elimina cada paso. |
> | **Qué se descartó** | Una tabla de conteos por paso: el descenso no se ve. |
> | **Cómo se lee** | Cada paso borra un color. Con los cinco, la barra desaparece: la tabla quedó limpia. |
> | **En Excel 365** | En el Editor de Power Query, haz clic en cada paso del panel Pasos aplicados y mira la vista previa. |

In [ ]:
# @title Gráfico 1 · Problemas que quedan después de cada paso
pasos_aplicados = 5  # @param {type:"slider", min:0, max:5, step:1}
limpio, evolucion = aplicar_pasos(erp, pasos_aplicados)
tabla = pd.DataFrame({n: p for n, p, _ in evolucion}).T
colores = [AZUL, MAGENTA, TEAL, "#7A86E8", GRIS]
fig = go.Figure()
for col, color in zip(tabla.columns, colores):
    fig.add_bar(name=col, y=tabla.index, x=tabla[col], orientation="h", marker_color=color, marker_line_color="white", marker_line_width=1)
quedan = int(tabla.iloc[-1].sum())
fig.update_layout(barmode="stack", height=320 + 30 * len(tabla), yaxis=dict(autorange="reversed"), xaxis_title="Celdas o filas con problema",
                  legend=dict(orientation="h", y=-0.25), title=f"Del origen a «{tabla.index[-1]}»: quedan {miles(quedan)} problemas")
fig.show()

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Cuántas filas toca cada paso? |
> | **Qué se grafica** | Barras: filas que cambian (o se quitan) en cada paso |
> | **Por qué esta forma** | Muestra el trabajo que hace cada paso: es lo que harías a mano cada mes. |
> | **Qué se descartó** | Porcentajes: para estimar esfuerzo importa el número de filas. |
> | **Cómo se lee** | El paso más largo es el que más trabajo manual reemplaza. |
> | **En Excel 365** | En Power Query: Inicio › Mantener filas o Ver › Perfil de columna para contar valores antes y después. |

In [ ]:
# @title Gráfico 2 · Filas que cambia cada paso
_, completo = aplicar_pasos(erp)
cambios = pd.Series({n: int(completo[i][1].sum() - completo[i + 1][1].sum()) for i, (n, _, _) in enumerate(completo[1:])})
cambios = cambios.sort_values()
fig = go.Figure(go.Bar(x=cambios.values, y=cambios.index, orientation="h", marker_color=AZUL,
                       text=[miles(v) for v in cambios.values], textposition="outside", cliponaxis=False))
mayores = cambios[cambios == cambios.max()].index.tolist()
fig.update_layout(title=f"«{cambios.idxmin()}» corrige {miles(cambios.min())} problemas; " + " y ".join(f"«{m}»" for m in mayores) + f", {miles(cambios.max())} cada uno",
                  xaxis_title="Problemas corregidos", height=320)
dar_aire(fig, cambios.max())
fig.show()
display(Markdown(f"Filas al final: **{miles(len(limpio))}** de {miles(len(erp))}. Las que faltan eran duplicadas."))

## 2 · El valor de repetir

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Cuánto tiempo cuesta limpiar a mano frente a actualizar? |
> | **Qué se grafica** | Líneas: horas acumuladas en un año con limpieza manual y con Power Query |
> | **Por qué esta forma** | Es una acumulación en el tiempo: las líneas muestran cuándo se recupera el tiempo invertido en armar la consulta. |
> | **Qué se descartó** | Una cifra de horas ahorradas: no se ve el punto en que la consulta empieza a pagar. |
> | **Cómo se lee** | Al principio Power Query cuesta más (armar la consulta). Desde el cruce de las líneas, todo es ahorro. |
> | **En Excel 365** | Datos › Consultas y conexiones › clic derecho › Propiedades › Actualizar al abrir el archivo. |

In [ ]:
# @title Gráfico 3 · Horas acumuladas en un año
minutos_manual = 40  # @param {type:"slider", min:10, max:120, step:5}
veces_por_mes = 4  # @param [1, 2, 4, 20]
horas_armar_consulta = 3  # @param [1, 2, 3, 5]
meses = np.arange(0, 13)
manual = meses * veces_por_mes * minutos_manual / 60
pq = horas_armar_consulta + meses * veces_por_mes * 1 / 60
fig = go.Figure()
fig.add_scatter(x=meses, y=manual, name="Limpieza manual", line=dict(color=MAGENTA, width=3))
fig.add_scatter(x=meses, y=pq, name="Power Query (1 min por actualización)", line=dict(color=AZUL, width=3))
cruce = next((m for m in meses if manual[m] > pq[m]), None)
fig.update_layout(title=f"En un año: {manual[-1]:.0f} h a mano frente a {pq[-1]:.0f} h con Power Query"
                  + (f" · se recupera en el mes {cruce}" if cruce else ""),
                  xaxis_title="Meses", yaxis_title="Horas acumuladas", height=360, legend=dict(orientation="h", y=-0.2))
fig.show()

## 3 · Retos de interpretación

In [ ]:
# @title Reto 1 · Según el gráfico 2, ¿qué paso corrige menos problemas?
respuesta = "Elige una opción"  # @param ["Elige una opción", "Recortar y MAYÚSCULAS en Cod Cliente", "Limpiar Valor Neto ($ y puntos)", "Limpiar Cantidad (quitar und)", "Convertir Fecha (try … otherwise)", "Quitar duplicados"]
revisar(respuesta, cambios.idxmin(), """Es la barra más corta del gráfico 2. Aun así, encontrar duplicados a mano es de lo más lento.""")

In [ ]:
# @title Reto 2 · ¿Cuántas filas duplicadas quita el último paso?
respuesta = "Elige una opción"  # @param ["Elige una opción", "27", "54", "81", "108"]
revisar(respuesta, miles(int(erp.duplicated().sum())), """Es la diferencia entre las filas del origen y las del resultado.""")

In [ ]:
# @title Reto 3 · Llega la exportación de febrero con los mismos defectos. ¿Qué haces en Excel?
respuesta = "Elige una opción"  # @param ["Elige una opción", "Repetir los pasos uno por uno", "Reemplazar el origen y usar Datos › Actualizar todo", "Crear una consulta nueva", "Limpiar solo las filas nuevas"]
revisar(respuesta, "Reemplazar el origen y usar Datos › Actualizar todo", """Los pasos quedaron guardados en la consulta: Actualizar los repite sobre los datos nuevos.""")

## Apóyate en IA
Para entender un paso que no conoces:

```text
En Power Query de Excel 365 tengo este paso:
= Table.ReplaceValue(#"Texto recortado","$","",Replacer.ReplaceText,{"Valor Neto"})
Explícame qué hace en palabras simples, cómo se hace con los menús y por qué debe ir antes de cambiar el tipo a número.
```

> **Verifica siempre.** Compara la respuesta de la IA con un dato que ya conozcas. No pegues datos personales reales (Ley 1581 de 2012): estos datos son ficticios, los de tu empresa no.

## Resumen
* Power Query guarda cada paso de limpieza; los datos originales no cambian.
* Borra el paso automático «Tipo cambiado» y fija los tipos al final.
* Lo que se repite cada mes se arma una vez y se actualiza con un clic.

**Siguiente:** [Clave 9 · Datos que se validan solos](https://omarli-360.github.io/excel-corporativo-12-claves/#clave-09) · [Cuaderno 09](https://colab.research.google.com/github/OmarLi-360/excel-corporativo-12-claves/blob/main/notebooks/09_validacion_y_formato.ipynb)

<sub>Dr. Bits · ADNBITS S.A.S. · Contenido CC BY-SA 4.0 · Datos ficticios de Distribuidora Andina S.A.S.</sub>